# Advanced Mid-Retrieval: Smart Orchestration

#### You've Mastered the Building Blocks... Now What?

In **Tutorial 1A**, you learned how to chunk documents effectively.  
In **Tutorial 1B**, you learned how to optimize queries intelligently.  
In **Tutorial 2A**, you learned 4 core retrieval techniques: Dense, Hybrid, Reranking, and Parent-Child.

Now comes the game-changer: **Combining these techniques intelligently!**

---

## The Journey So Far

```
Tutorial 1A: Chunking ✅
    ↓
Tutorial 1B: Query Optimization ✅
    ↓
Tutorial 2A: Core Retrieval Methods ✅
    ↓
Tutorial 2B: Smart Orchestration ← You are here
    ↓
Next: Post Retrieval Strategies
```

**You've learned the individual tools. Now let's orchestrate them!**

---

## The Challenge

You now have powerful tools from 2A:
- ✅ Dense Search (fast, semantic)
- ✅ Hybrid Search (keywords + semantics)
- ✅ Reranking (precision boost)
- ✅ Parent-Child (context optimization)

**But here's the problem:**

- Using **Hybrid + Reranking** for "Hello!" wastes resources
- Using **Dense-only** for complex queries misses relevant content
- Different queries need **different approaches**

**The question:** How do we choose the right technique for each query?

---

## What You'll Learn

We'll explore **3 advanced orchestration techniques** that intelligently use the methods from 2A:

**The Techniques:**
1. **Multi-Query Retrieval** - Generate query variants for broader coverage
2. **Query Routing** - Auto-select the right method per query type
3. **Adaptive Retrieval** - Auto-configure based on query complexity

**Plus:** How to combine everything into production pipelines!

---

## Tutorial Structure

```
Part 1: Multi-Query (variant generation + fusion)
    ↓
Part 2: Query Routing (auto-select methods)
    ↓
Part 3: Adaptive Retrieval (auto-configure parameters)
    ↓
Part 4: Production Pipelines (combine everything)
    ↓
Part 5: Decision Guide & Summary
```

**Our approach:** Learn smart orchestration, see it in action, then build production pipelines.

Let's orchestrate! 🚀

## Setup: Connect to Vector Database

In [1]:
import warnings
import logging
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

from retrieval_playground.utils import config
from retrieval_playground.utils.model_manager import model_manager
from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever
from retrieval_playground.src.mid_retrieval.multi_query_hybrid import MultiQueryHybrid
from retrieval_playground.src.mid_retrieval.adaptive_retrieval import AdaptiveRetriever
from retrieval_playground.src.mid_retrieval.route_driven_retrieval import RouteRetriever

import warnings
warnings.filterwarnings('ignore')

# Connect to Qdrant
qdrant_client = QdrantClient(url=config.QDRANT_URL, api_key=config.QDRANT_KEY)
embeddings = model_manager.get_embeddings()

# Create vector store (using recursive_character from 2A)
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="recursive_character",
    embedding=embeddings
)

print("✅ Setup complete!")
print("   Collection: recursive_character")
print("   Ready to explore advanced orchestration!")

✅ Setup complete!
   Collection: recursive_character
   Ready to explore advanced orchestration!


---

# Technique 5: Multi-Query Retrieval

## The Problem: Vocabulary Mismatch

Even with semantic search, a **single query** can miss relevant documents:

**Example:**
- **Your query:** "How do neural networks learn?"
- **Missed docs using:** "training process", "backpropagation", "gradient descent", "optimization algorithms"

Why? Dense search creates ONE vector from ONE phrasing. Different vocabulary = different embeddings.

---

## The Solution: Multi-Query Pipeline

Our implementation combines **FOUR stages** for maximum quality:

```
Original Query: "How do neural networks learn?"
                            ↓
        ┌─────────────────────────────────────────┐
        │    Stage 1: Generate Variants (LLM)     │
        └─────────────────────────────────────────┘
                            ↓
    ┌────────────────┬────────────────┬────────────────┐
    ↓                ↓                ↓                ↓
Variant 1:     Variant 2:       Variant 3:       Original:
"Neural net    "How does        "Learning        "How do NNs
training       backprop         mechanisms in     learn?"
process"       work in NNs?"    deep learning"
    ↓                ↓                ↓                ↓
        ┌─────────────────────────────────────────┐
        │    Stage 2: Hybrid Search (BM25+Dense)  │
        │     Run for EACH variant (4 searches)   │
        └─────────────────────────────────────────┘
    ↓                ↓                ↓                ↓
    └────────────────┴────────────────┴────────────────┘
                            ↓
       ┌─────────────────────────────────────────┐
       │           Stage 3: RRF Fusion           │
       │         Merge all 4 result sets         │
       └─────────────────────────────────────────┘
                            ↓
       ┌─────────────────────────────────────────┐
       │   Stage 4: Reranking (Cross-Encoder)    │
       │    Top n candidates → Top 5 precise     │
       └─────────────────────────────────────────┘
                            ↓
                      Final Results
                    (Maximum Quality!)
```

**This combines THREE techniques:**
1. Hybrid Search (BM25 + Dense + RRF)
2. Reranking (cross-encoder precision)
3. Multi-Query orchestration

---

<img src="../../data/display_images/2B_multi_query_hybrid.png" style="width: 60%; max-width: 700px;">

  **[AI Generated Image]**

## How It Works: Step by Step

**Stage 1: Generate Variants (Using LLM)**
```
Prompt to LLM: "Generate 3 alternative phrasings of: {query}"
               "Use different vocabulary but keep the same meaning"

Output: 3 query variants with diverse vocabulary
```

**Stage 2: Hybrid Search for Each Variant**
```
4 searches total (each is BM25 + Dense + RRF):
  - Original query → Hybrid results set 1
  - Variant 1 → Hybrid results set 2
  - Variant 2 → Hybrid results set 3
  - Variant 3 → Hybrid results set 4
```

**Stage 3: Merge with RRF**
```
Same RRF formula from Hybrid Search:

RRF_score = 1/(60 + rank_v1) + 1/(60 + rank_v2) + 
            1/(60 + rank_v3) + 1/(60 + rank_original)

Documents appearing in MULTIPLE variant results score higher!
```

**Stage 4: Rerank with Cross-Encoder**
```
Take top n candidates from RRF fusion
  ↓
Pass to cross-encoder
  ↓
Get precise relevance scores (0-1)
  ↓
Return top 5 sorted by rerank_score
```

---

## Understanding the Scores

**After Stage 3 (RRF):** Documents have `rrf_score` (0-0.035 range)  
**After Stage 4 (Reranking):** Documents have `rerank_score` (0-1 range) ← **FINAL SORTING**

**Final results are sorted by `rerank_score`**, not `rrf_score`!

Cross-encoder gives the final word on relevance!

---

## When Multi-Query Pipeline Shines

**✅ Perfect for:**
- Complex questions with multiple valid phrasings
- Technical domains where terminology varies ("ML training" vs "model optimization")
- Broad exploratory questions ("How does X work?")
- Research queries needing comprehensive coverage
- High-stakes applications (medical, legal, financial)

**❌ Not ideal for:**
- Simple factual lookups ("What is BERT?") - overkill
- Queries with very specific unique terms 
- When speed is critical (4x searches + reranking = ~400-500ms)
- Low-cost deployments (4x search cost)

### 🔬 Try It Yourself - See the Variants

Let's first see what query variants look like:

In [2]:
from retrieval_playground.src.pre_retrieval.query_rephrasing import expand_query

# Example query
query = "How do scientific agents plan and decompose complex research tasks?"

# Generate variants
variants = expand_query(query, num_variants=3)

print("MULTI-QUERY VARIANT GENERATION")
print("=" * 80)
print(f"\nOriginal Query:")
print(f"   {query}\n")

print("Generated Variants:")
for i, variant in enumerate(variants, 1):
    print(f"   {i}. {variant.replace(chr(10), ' ')}")

print("\n💡 Notice how each variant uses different vocabulary!")
print("=" * 80)

MULTI-QUERY VARIANT GENERATION

Original Query:
   How do scientific agents plan and decompose complex research tasks?

Generated Variants:
   1. What is the methodology behind how autonomous scientific agents break down intricate research objectives into manageable steps?
   2. Explain the task decomposition and planning frameworks utilized by AI agents in scientific research workflows.
   3. Scientific agent task planning and decomposition strategies for complex research.

💡 Notice how each variant uses different vocabulary!


### 🔬 Try It Yourself - Multi-Query in Action

Now let's use Multi-Query with Hybrid Search:

In [3]:
# Initialize multi-query retriever (uses Hybrid Search + Reranking internally)
mqh = MultiQueryHybrid(collection_name="hybrid")

# Search using query variants
query = "How do scientific agents plan and decompose complex research tasks?"
results = mqh.search(query, k=5)

print("MULTI-QUERY HYBRID RESULTS")
print("=" * 80)
print(f"Query: '{query}'\n")
print("Full Pipeline (4 stages):")
print("  1. Generated 3 query variants (different vocabulary)")
print("  2. Ran Hybrid Search (BM25 + Dense) for each variant")
print("  3. Merged all results using RRF")
print("  4. Reranked top k with cross-encoder (returns top 5)\n")

for i, doc in enumerate(results, 1):
    # After reranking, results have rerank_score (sorted by this!)
    rerank_score = doc.metadata.get('rerank_score', 0)
    print(f"{i}. Rerank Score: {rerank_score:.4f}")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:120].replace(chr(10), ' ')}...\n")

print("=" * 80)
print("💡 This is the FULL pipeline combining THREE techniques:")
print("   - Multi-Query (variant generation)")
print("   - Hybrid Search (BM25 + Dense + RRF)")
print("   - Reranking (cross-encoder precision)")
print("   Result: Maximum quality!")
print("=" * 80)

mqh.close()

✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Multi-query hybrid initialized (3 variants, FlashRank reranker)
MULTI-QUERY HYBRID RESULTS
Query: 'How do scientific agents plan and decompose complex research tasks?'

Full Pipeline (4 stages):
  1. Generated 3 query variants (different vocabulary)
  2. Ran Hybrid Search (BM25 + Dense) for each variant
  3. Merged all results using RRF
  4. Reranked top k with cross-encoder (returns top 5)

1. Rerank Score: 0.9958
   Source: 2025_Scientific_Intelligence_Survey.pdf
   Preview: 2 Architecture The architecture of LLM-based scientific agents is designed to enable iterative, context-aware process- i...

2. Rerank Score: 0.9939
   Source: 2025_Scientific_Intelligence_Survey.pdf
   Preview: sequences that orchestrate tool invocations, mem- ory operations, and verification steps. In au- tonomous scientific dis...

3. Rerank Score: 0.9908
   Source: 2025_Scientific_Intelligence_Survey.pdf
   Preview: evaluation metrics and expanding bench

---

# Technique 6: Query Routing

## The Problem: One Size Doesn't Fit All

Not all queries need the same retrieval approach:

**Examples:**
- **"Hello!"** → No retrieval needed (greeting)
- **"What is BERT?"** → Hybrid Search (factual lookup)
- **"Compare BERT and GPT-3 architectures"** → Multi-Query (needs breadth)
- **"How do transformers handle long sequences?"** → Hybrid Search (analytical)

Using Multi-Query for "Hello!" wastes resources.  
Using Dense-only for comparisons misses relevant content.

---

## The Solution: Semantic Routing

**Automatically detect query type** and select the appropriate method:

```
              User Query
                  ↓
   Semantic Router (analyzes intent)
                  ↓
  ┌──────────┬──────────┬──────────┐
  ↓          ↓          ↓          ↓         
Greeting  Factual   Comparison  Analytical   
  ↓          ↓          ↓          ↓          
No         Hybrid    Multi-     Hybrid  
Retrieval  Search    Query      Search
```

---

## How It Works: Semantic Routing

**Step 1: Define Routes**
```python
routes = [
    Route(
        name="greeting",
        utterances=["hello", "hi", "hey", "good morning"],
        requires_retrieval=False
    ),
    Route(
        name="factual",
        utterances=["what is", "define", "explain briefly"],
        method="hybrid_search"
    ),
    Route(
        name="comparison",
        utterances=["compare", "difference between", "X vs Y"],
        method="multi_query"
    ),
    Route(
        name="analytical",
        utterances=["how does X work", "explain the process"],
        method="hybrid_search"
    )
]
```

**Step 2: Route Query**
```
Query: "Compare BERT and GPT-3"
  ↓
Router embeds query and utterances
  ↓
Finds closest match: "comparison" route
  ↓
Returns: method="multi_query"
  ↓
Executes Multi-Query Hybrid
```

**Step 3: Execute Method**
```python
if route.requires_retrieval == False:
    return []  # No retrieval needed
elif route.method == "hybrid_search":
    return hybrid_search(query)
elif route.method == "multi_query":
    return multi_query_search(query)
```

---

## Benefits of Routing

**1. Efficiency** 
- Greetings skip retrieval entirely
- Simple queries use Hybrid Search
- Comparison queries get Multi-Query (broader coverage)

**2. Quality** 
- Method matches query type
- Comparisons get breadth (Multi-Query)
- Analytical questions get good balance (Hybrid)

**3. Cost Savings**
- Don't waste Multi-Query on simple factual lookups

---

## When to Use Routing

**✅ Perfect for:**
- Production chatbots with varied query types
- Applications with mix of simple/complex queries
- Cost-sensitive deployments
- Multi-domain systems

**❌ Not needed for:**
- Single query type (all factual or all complex)
- Research/prototyping (just use best method)
- Very small scale (routing overhead not worth it)

### 🔬 Try It Yourself - See Routing in Action

In [4]:
# Initialize route-driven retriever
route_retriever = RouteRetriever(collection_name="hybrid")

print("QUERY ROUTING - Example 1: Greeting")
print("=" * 80)

query = "Hello!"
print(f"\nQuery: '{query}'")

# Get route info first
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Requires Retrieval: {route_info['route']['requires_retrieval']}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\nProcessing 🔄")

# Execute search
results = route_retriever.search(query)
print(f"\nResults: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("=" * 80)

✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Route-driven retriever initialized
QUERY ROUTING - Example 1: Greeting

Query: 'Hello!'

📍 Route Classification:
   Detected Type: GREETINGS
   Requires Retrieval: False
   Complexity: simple

Processing 🔄

Results: 0 documents


In [5]:
print("QUERY ROUTING - Example 2: Factual Question")
print("=" * 80)

query = "What is BERT?"
print(f"\nQuery: '{query}'")

# Get route info
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Selected Method: {route_info['final_retrieval_method'].upper()}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\nProcessing 🔄")

# Execute search
results = route_retriever.search(query)

print(f"\nResults: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("=" * 80)

QUERY ROUTING - Example 2: Factual Question

Query: 'What is BERT?'

📍 Route Classification:
   Detected Type: FACTUAL
   Selected Method: HYBRID_SEARCH
   Complexity: simple

Processing 🔄

Results: 5 documents

   Top Result:
   Sabater, J. Nicolas, C. Peubey, R. Radu, D. Scheperset al., “The era5 global reanalysis,”Quarterly J...


In [6]:
print("QUERY ROUTING - Example 3: Comparison Question")
print("=" * 80)

query = "Compare BERT and GPT-3 architectures"
print(f"\nQuery: '{query}'")

# Get route info
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Selected Method: {route_info['final_retrieval_method'].upper()}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\nProcessing 🔄")
print(f"   Stage 1: Generate 3 query variants (LLM)")

# Generate variants to show
from retrieval_playground.src.pre_retrieval.query_rephrasing import expand_query
variants = expand_query(query, num_variants=3)
for i, variant in enumerate(variants, 1):
    print(f"      Variant {i}: {variant[:60].replace(chr(10), ' ')}...")

# Execute search
results = route_retriever.search(query)

print(f"\nResults: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("=" * 80)

QUERY ROUTING - Example 3: Comparison Question

Query: 'Compare BERT and GPT-3 architectures'

📍 Route Classification:
   Detected Type: COMPARISON
   Selected Method: MULTI_QUERY
   Complexity: moderate

Processing 🔄
   Stage 1: Generate 3 query variants (LLM)
      Variant 1: What are the primary architectural differences between BERT ...
      Variant 2: A comparative analysis of the transformer structures used in...
      Variant 3: BERT vs GPT-3 model design: structural comparison and techni...

Results: 8 documents

   Top Result:
   and ChatGPT (Achiam et al. (2023); Hurst et al. (2024); OpenAI (2022)), leverage vast datasets and s...


In [7]:
print("QUERY ROUTING - Example 4: Complex Question")
print("=" * 80)

query = "How do attention mechanisms work in transformers?"
print(f"\nQuery: '{query}'")

# Get route info
route_info = route_retriever.get_route_info(query)
print(f"\n📍 Route Classification:")
print(f"   Detected Type: {route_info['route']['route_name'].upper()}")
print(f"   Selected Method: {route_info['final_retrieval_method'].upper()}")
print(f"   Complexity: {route_info['complexity']['complexity']}")

print(f"\nProcessing 🔄")


# Execute search
results = route_retriever.search(query)

print(f"\nResults: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("\n" + "=" * 80)

route_retriever.close()

QUERY ROUTING - Example 4: Complex Question

Query: 'How do attention mechanisms work in transformers?'

📍 Route Classification:
   Detected Type: ANALYTICAL
   Selected Method: HYBRID_SEARCH
   Complexity: complex

Processing 🔄

Results: 8 documents

   Top Result:
   z′ ℓ =MSA(LN(z ℓ−1)) +zℓ−1, ℓ=1, . . . ,L, zℓ,=MLP   LN   z′ ℓ  +z ′ ℓ, ℓ=1, . . . ,L, where MSA i...



---

# Technique 7: Adaptive Retrieval

## The Problem: Fixed Parameters Waste Resources

In 2A, we used fixed parameters:
- `k=5` (always return 5 results)
- Always same method (Dense, Hybrid, etc.)
- Same threshold for all queries

But different queries have different needs:

**Simple query:**  
"What is BERT?" → Needs k=3, fast Dense Search

**Complex query:**  
"How do attention mechanisms work?" → Needs k=8, Multi-Query + Reranking

**The waste:**  
Using k=8 for "What is BERT?" returns 5 irrelevant docs  
Using k=3 for complex questions misses important content

---

## The Solution: Adaptive Configuration

**Automatically analyze query complexity** and adjust parameters:

```
           User Query
               ↓
      Complexity Analyzer
               ↓
  ┌────────────┬────────────┐
  ↓            ↓            ↓            
Simple      Moderate      Complex      
  ↓            ↓            ↓         
 k=3          k=5          k=8        
Dense       Hybrid      Multi-Query
            +Rerank      +Rerank    
```

---

## How It Works: Complexity Analysis

**Step 1: Analyze Query**
```python
def analyze_complexity(query):
    score = assign_complexity_score(query)
    
    # Classify
    if score <= 1: return 'simple'
    elif score == 2: return 'moderate'
    else: return 'complex'  # score 3+
```

**Step 2: Configure Parameters**
```python
config_map = {
    'simple': {
        'k': 3,
        'method': 'dense',
        'use_reranking': False
    },
    'moderate': {
        'k': 5,
        'method': 'hybrid',
        'use_reranking': True
    },
    'complex': {
        'k': 8,
        'method': 'multi_query',
        'use_reranking': True
    }
}
```

**Step 3: Execute with Config**
```python
complexity = analyze_complexity(query)
config = config_map[complexity]
results = retrieve(query, **config)
```

---

## Benefits of Adaptive Retrieval

**1. Efficiency** 
- Simple queries use minimal resources (k=3, Dense)
- Complex queries get full power (k=8, Multi-Query + Rerank)

**2. Quality** 
- Simple queries avoid noise (fewer irrelevant results)
- Complex queries get breadth (more candidates)

**3. Automatic** 
- No manual tuning per query
- Self-adjusting system

---

## When to Use Adaptive Retrieval

**✅ Perfect for:**
- Production systems with varied query complexity
- Unknown query patterns (can't predict difficulty)
- Cost-sensitive deployments
- "Set and forget" solutions

**❌ Not needed for:**
- Homogeneous query complexity (all simple or all complex)
- When you want explicit control
- Very small scale (complexity analysis overhead)

### 🔬 Try It Yourself - See Adaptive Retrieval in Action

In [8]:
# Initialize adaptive retriever
adaptive = AdaptiveRetriever(collection_name="hybrid")

print("ADAPTIVE RETRIEVAL - Example 1: Simple Query")
print("=" * 80)

query = "What is BERT?"
print(f"\nQuery: '{query}'")

# Get complexity analysis
complexity_info = adaptive.get_complexity_info(query)
print(f"\n📍 Complexity Analysis:")
print(f"   Complexity Level: {complexity_info['complexity']['complexity'].upper()}")
print(f"   Complexity Score: {complexity_info['complexity']['score']}")
print(f"   Selected Method: {complexity_info['config']['method'].upper()}")
print(f"   k (results): {complexity_info['config']['k']}")
print(f"   Reranking: {complexity_info['config']['use_reranking']}")

print(f"\nProcessing 🔄")

# Execute search
results = adaptive.search(query)

print(f"\nResults: {len(results)} documents")
if results:
    print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("=" * 80)

✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Multi-query hybrid initialized (3 variants, FlashRank reranker)
✅ Adaptive retriever initialized
ADAPTIVE RETRIEVAL - Example 1: Simple Query

Query: 'What is BERT?'

📍 Complexity Analysis:
   Complexity Level: SIMPLE
   Complexity Score: 0
   Selected Method: DENSE
   k (results): 3
   Reranking: False

Processing 🔄

Results: 3 documents

   Top Result:
   labels) that differ by only a few characters but carry distinct physical meaning, whereas BM25 retri...


In [9]:
print("ADAPTIVE RETRIEVAL - Example 2: Moderate Query")
print("=" * 80)

query = "Compare BERT and GPT-3"
print(f"\nQuery: '{query}'")

# Get complexity analysis
complexity_info = adaptive.get_complexity_info(query)
print(f"\n📍 Complexity Analysis:")
print(f"   Complexity Level: {complexity_info['complexity']['complexity'].upper()}")
print(f"   Complexity Score: {complexity_info['complexity']['score']}")
print(f"   Selected Method: {complexity_info['config']['method'].upper()}")
print(f"   k (results): {complexity_info['config']['k']}")
print(f"   Reranking: {complexity_info['config']['use_reranking']}")

print(f"\nProcessing 🔄")

# Execute search
results = adaptive.search(query)

print(f"\nResults: {len(results)} documents")
if results:
    # Moderate queries use reranking
    rerank_score = results[0].metadata.get('rerank_score', 0)
    if rerank_score:
        print(f"\n   Top Result (Rerank Score: {rerank_score:.4f}):")
    else:
        print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("=" * 80)

ADAPTIVE RETRIEVAL - Example 2: Moderate Query

Query: 'Compare BERT and GPT-3'

📍 Complexity Analysis:
   Complexity Level: MODERATE
   Complexity Score: 2
   Selected Method: HYBRID
   k (results): 5
   Reranking: True

Processing 🔄

Results: 5 documents

   Top Result (Rerank Score: 0.5142):
   average overall ratings ranging from 3.5/10 with gpt-4o, 3.8/10 with o1-mini, and 4.0/10 with o1-pre...


In [10]:
print("ADAPTIVE RETRIEVAL - Example 3: Complex Query")
print("=" * 80)

query = "How do attention mechanisms work in transformers?"
print(f"\nQuery: '{query}'")

# Get complexity analysis
complexity_info = adaptive.get_complexity_info(query)
print(f"\n📍 Complexity Analysis:")
print(f"   Complexity Level: {complexity_info['complexity']['complexity'].upper()}")
print(f"   Complexity Score: {complexity_info['complexity']['score']}")
print(f"   Selected Method: {complexity_info['config']['method'].upper()}")
print(f"   k (results): {complexity_info['config']['k']}")
print(f"   Reranking: {complexity_info['config']['use_reranking']}")

print(f"\nProcessing 🔄")
if complexity_info['config']['method'] == 'multi_query':
    print(f"   Stage 1: Generate 3 query variants (LLM)")
    print(f"   Stage 2: Run Hybrid Search for each variant")
    print(f"   Stage 3: Merge results with RRF fusion")
    print(f"   Stage 4: Rerank for precision")

# Execute search
results = adaptive.search(query)

print(f"\nResults: {len(results)} documents")
if results:
    # Complex queries use multi-query + reranking
    rerank_score = results[0].metadata.get('rerank_score', 0)
    if rerank_score:
        print(f"\n   Top Result (Rerank Score: {rerank_score:.4f}):")
    else:
        print(f"\n   Top Result:")
    print(f"   {results[0].page_content[:100].replace(chr(10), ' ')}...")

print("\n" + "=" * 80)

adaptive.close()

ADAPTIVE RETRIEVAL - Example 3: Complex Query

Query: 'How do attention mechanisms work in transformers?'

📍 Complexity Analysis:
   Complexity Level: COMPLEX
   Complexity Score: 3
   Selected Method: MULTI_QUERY
   k (results): 8
   Reranking: True

Processing 🔄
   Stage 1: Generate 3 query variants (LLM)
   Stage 2: Run Hybrid Search for each variant
   Stage 3: Merge results with RRF fusion
   Stage 4: Rerank for precision

Results: 8 documents

   Top Result (Rerank Score: 0.0563):
   processing systems35, 36479–36494 (2022). 8. Dosovitskiy, A.et al.An image is worth 16x16 words: Tra...



---

# 🚀 Building Production Pipelines

Now that you've learned all 7 techniques, let's see how to **combine** them in production!

---

## 4 Common Pipeline Patterns

| Pipeline | Best For | Techniques | Performance |
|----------|----------|------------|-------------|
| **Speed** | High-throughput apps | Adaptive only | ~50ms, +20-25% quality |
| **Balanced** | General production | Routing + Hybrid + Rerank | ~150ms, +30-35% quality |
| **Quality** | High-stakes (medical, legal) | Multi-Query + Hybrid + Rerank | ~500ms, +45-60% quality |
| **Smart** | Varied queries | Routing + Adaptive | ~100ms, +30-50% quality |

---

## When to Use Each

**Start with Balanced**, then adjust:
- Too slow? → Switch to **Speed**
- Quality insufficient? → Upgrade to **Quality**  
- Unknown query patterns? → Switch to **Smart**

**Example:**
```python
# Balanced Pipeline (recommended starting point)
query → routing → hybrid_search → reranking → top_5

# If latency is critical
query → adaptive → (dense OR hybrid based on complexity)

# If quality is critical
query → multi_query → hybrid_search → reranking → top_3
```

### 🔬 Try It Yourself -  Compare Pipelines
Let's compare different pipeline configurations:

In [11]:
import time
from retrieval_playground.src.mid_retrieval.reranking import Reranker

query = "How do attention mechanisms work in transformer neural networks?"

print("🚀 PIPELINE COMPARISON")
print("=" * 80)
print(f"Query: '{query}'\n")

# Pipeline 1: Dense Only (baseline from 2A)
print("1️⃣  BASELINE - Dense Only")
start = time.time()
results_dense = vector_store.similarity_search_with_score(query, k=5)
latency_dense = (time.time() - start) * 1000
top_score_dense = results_dense[0][1] if results_dense else 0
print(f"   Latency: {latency_dense:.0f}ms")
print(f"   Top score: {top_score_dense:.4f}")
print(f"   Preview: {results_dense[0][0].page_content[:60].replace(chr(10), ' ')}...\n")

# Pipeline 2: Hybrid Search
print("2️⃣  HYBRID - BM25 + Dense + RRF")
hybrid = HybridRetriever(collection_name="hybrid")
start = time.time()
results_hybrid = hybrid.search(query, k=5)
latency_hybrid = (time.time() - start) * 1000
top_score_hybrid = results_hybrid[0].metadata.get('rrf_score', 0) if results_hybrid else 0
print(f"   Latency: {latency_hybrid:.0f}ms")
print(f"   Top RRF score: {top_score_hybrid:.4f}")
print(f"   Preview: {results_hybrid[0].page_content[:60].replace(chr(10), ' ')}...\n")
hybrid.close()

# Pipeline 3: Hybrid + Reranking (Dense → Rerank)
print("3️⃣  BALANCED - Dense → Reranking")
reranker = Reranker(collection_name="recursive_character", top_k=20, top_n=5)
start = time.time()
results_rerank = reranker.retrieve(query)
latency_rerank = (time.time() - start) * 1000
top_score_rerank = results_rerank[0].metadata.get('rerank_score', 0) if results_rerank else 0
print(f"   Latency: {latency_rerank:.0f}ms")
print(f"   Top rerank score: {top_score_rerank:.4f}")
print(f"   Preview: {results_rerank[0].page_content[:60].replace(chr(10), ' ')}...\n")
reranker.close()

# Pipeline 4: Multi-Query Hybrid (Full pipeline with reranking)
print("4️⃣  QUALITY - Multi-Query + Hybrid + Reranking")
mqh = MultiQueryHybrid(collection_name="hybrid")
start = time.time()
results_mqh = mqh.search(query, k=5)
latency_mqh = (time.time() - start) * 1000
# Multi-query returns reranked results
top_score_mqh = results_mqh[0].metadata.get('rerank_score', 0) if results_mqh else 0
print(f"   Latency: {latency_mqh:.0f}ms")
print(f"   Top rerank score: {top_score_mqh:.4f}")
print(f"   Preview: {results_mqh[0].page_content[:60].replace(chr(10), ' ')}...\n")
mqh.close()

print("=" * 80)
print("SUMMARY:")
print(f"   Baseline (Dense):              {latency_dense:.0f}ms")
print(f"   Hybrid (BM25+Dense):           {latency_hybrid:.0f}ms (+{latency_hybrid/latency_dense:.1f}x)")
print(f"   Balanced (Dense→Rerank):       {latency_rerank:.0f}ms (+{latency_rerank/latency_dense:.1f}x)")
print(f"   Quality (Multi→Hybrid→Rerank): {latency_mqh:.0f}ms (+{latency_mqh/latency_dense:.1f}x)")
print("\nTrade-off: Higher latency → Better quality")
print("   Pipelines 3 & 4 are sorted by rerank scores (most precise!)")
print("   Choose based on your speed vs quality requirements!")
print("\n📌 Note: Rerank scores (0-1) reflect answer quality. Low scores (<0.5) mean")
print("   documents are topically related but don't directly answer the question.")
print("=" * 80)

🚀 PIPELINE COMPARISON
Query: 'How do attention mechanisms work in transformer neural networks?'

1️⃣  BASELINE - Dense Only
   Latency: 1460ms
   Top score: 0.7090
   Preview: z′ ℓ =MSA(LN(z ℓ−1)) +zℓ−1, ℓ=1, . . . ,L, zℓ,=MLP   LN   z′...

2️⃣  HYBRID - BM25 + Dense + RRF
✅ Hybrid retriever initialized (collection: 'hybrid')
   Latency: 2139ms
   Top RRF score: 0.0318
   Preview: z′ ℓ =MSA(LN(z ℓ−1)) +zℓ−1, ℓ=1, . . . ,L, zℓ,=MLP   LN   z′...

3️⃣  BALANCED - Dense → Reranking
   Latency: 2249ms
   Top rerank score: 0.2743
   Preview: with group normalization. The model uses an embedding dimens...

4️⃣  QUALITY - Multi-Query + Hybrid + Reranking
✅ Hybrid retriever initialized (collection: 'hybrid')
✅ Multi-query hybrid initialized (3 variants, FlashRank reranker)
   Latency: 9543ms
   Top rerank score: 0.2597
   Preview: with group normalization. The model uses an embedding dimens...

SUMMARY:
   Baseline (Dense):              1460ms
   Hybrid (BM25+Dense):           2139ms (+1.5x)
  

---

# Summary & Next Steps

## What You've Learned

**From 2A (Core Techniques):**
1. Dense Search - Baseline semantic similarity
2. Hybrid Search - BM25 + Dense + RRF fusion  
3. Reranking - Two-stage precision boost
4. Parent-Child - Adaptive context sizing

**From 2B (Orchestration):**
1. Multi-Query - Query variants + fusion for breadth
2. Query Routing - Auto-select method by query type
3. Adaptive Retrieval - Auto-configure by complexity

---

## Quick Decision Guide

| Your Need | Technique | Gain | Trade-off |
|-----------|-----------|------|-----------|
| Better keyword matching | Hybrid Search | +15-25% | ~100ms |
| More precision | Reranking | +10-15% | +50ms |
| Broader coverage | Multi-Query | +15-20% | +200ms, 4x cost |
| Varied query types | Routing | +15-20% | Minimal |
| Efficiency | Adaptive | +30% eff | Auto-scales cost |

---

## Recommended Path

### 1. **Start Here**
```
Hybrid Search (from 2A)
→ Good baseline for most apps (~100ms, +15-25%)
```

### 2. **Level Up** (choose ONE based on your needs)
```
• Need precision? → Add Reranking
• Varied queries? → Add Routing
• Unknown complexity? → Add Adaptive
• Need breadth? → Add Multi-Query
```

### 3. **Production Ready**
```
Combine into pipeline:
• General use: Routing + Hybrid + Rerank
• High quality: Multi-Query + Hybrid + Rerank
• Auto-optimize: Routing + Adaptive
```

---

## Pro Tips 💡

1. **Measure first** - Don't assume complex = better
2. **Start simple** - Add complexity only when needed
3. **Watch latency** - User experience matters
4. **Consider cost** - Multi-Query = 4x searches
5. **A/B test** - Validate with real queries

---

## Resources

- **Code:** `retrieval_playground/src/mid_retrieval/`

---


You now know:
- ✅ 4 core retrieval techniques
- ✅ 3 orchestration methods  
- ✅ How to combine them
- ✅ When to use each

**Next:** Tutorial 3 - Post-Retrieval Strategies

**You're ready to build production RAG systems!** 🚀